# Column Semantic Labeling — DistilBERT Fine-Tuning

**Goal:** Fine-tune `distilbert-base-uncased` to classify CSV column names into 10 semantic categories.

**Baseline:** Zero-shot classification with `typeform/distilbert-base-uncased-mnli`  
**After:** Fine-tuned `distilbert-base-uncased` on our synthetic column-name dataset

**10 Classes:** email, phone, name, date, currency, id, address, percentage, numeric, text

In [ ]:
import json
import os
import random
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, f1_score
)

import torch
from torch.utils.data import Dataset
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    TrainingArguments, Trainer,
    pipeline, EarlyStoppingCallback
)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

os.makedirs('../evaluation', exist_ok=True)
os.makedirs('../data/raw', exist_ok=True)

## 1. Build Synthetic Column Name Dataset

In [ ]:
LABEL_MAP = {
    'email': 0, 'phone': 1, 'name': 2, 'date': 3, 'currency': 4,
    'id': 5, 'address': 6, 'percentage': 7, 'numeric': 8, 'text': 9
}
ID2LABEL = {v: k for k, v in LABEL_MAP.items()}

COLUMN_EXAMPLES = {
    'email': [
        'email', 'user_email', 'email_address', 'customer_email',
        'contact_email', 'email_id', 'mail', 'e_mail', 'primary_email',
        'work_email', 'personal_email', 'registered_email', 'sender_email',
        'recipient_email', 'admin_email', 'support_email', 'billing_email',
        'notification_email', 'user_mail', 'account_email', 'login_email',
        'alt_email', 'secondary_email', 'email_contact', 'email_field',
        'emailaddress', 'useremail', 'contactmail', 'membermail', 'client_email',
        'vendor_email', 'partner_email', 'prof_email', 'office_email', 'team_email',
        'from_email', 'reply_email', 'confirm_email', 'verify_email', 'auth_email'
    ],
    'phone': [
        'phone', 'phone_number', 'mobile', 'telephone', 'contact_phone',
        'cell', 'cell_phone', 'mobile_number', 'tel', 'phone_no',
        'fax', 'fax_number', 'work_phone', 'home_phone', 'primary_phone',
        'secondary_phone', 'alt_phone', 'emergency_phone', 'business_phone',
        'office_phone', 'direct_phone', 'contact_number', 'whatsapp', 'phone_num',
        'mobile_no', 'tel_no', 'telephone_number', 'contact_mobile', 'cellular',
        'landline', 'phone_contact', 'sms_number', 'call_number', 'reach_phone',
        'phone_field', 'phonenumber', 'mobilenumber', 'customer_phone', 'user_phone',
        'staff_phone', 'vendor_phone'
    ],
    'name': [
        'name', 'full_name', 'first_name', 'last_name', 'customer_name',
        'user_name', 'username', 'display_name', 'given_name', 'surname',
        'family_name', 'middle_name', 'nickname', 'alias', 'person_name',
        'employee_name', 'staff_name', 'contact_name', 'owner_name', 'author_name',
        'company_name', 'business_name', 'vendor_name', 'client_name', 'account_name',
        'patient_name', 'student_name', 'member_name', 'agent_name', 'rep_name',
        'fname', 'lname', 'fullname', 'firstname', 'lastname',
        'legal_name', 'preferred_name', 'beneficiary_name', 'payee_name', 'sender_name'
    ],
    'date': [
        'date', 'created_at', 'updated_at', 'order_date', 'birth_date',
        'start_date', 'end_date', 'expiry_date', 'date_of_birth', 'dob',
        'join_date', 'signup_date', 'registration_date', 'purchase_date', 'ship_date',
        'delivery_date', 'due_date', 'invoice_date', 'payment_date', 'last_login',
        'timestamp', 'event_date', 'record_date', 'modified_date', 'posted_date',
        'effective_date', 'transaction_date', 'submitted_at', 'approved_at', 'closed_date',
        'open_date', 'hire_date', 'termination_date', 'review_date', 'created_date',
        'date_created', 'date_modified', 'date_joined', 'visit_date', 'scheduled_date'
    ],
    'currency': [
        'price', 'amount', 'total', 'cost', 'revenue',
        'salary', 'wage', 'fee', 'charge', 'payment',
        'balance', 'budget', 'expense', 'income', 'profit',
        'tax', 'discount', 'subtotal', 'grand_total', 'net_amount',
        'gross_amount', 'unit_price', 'list_price', 'sale_price', 'original_price',
        'total_amount', 'paid_amount', 'refund_amount', 'transaction_amount', 'invoice_amount',
        'monthly_salary', 'annual_income', 'total_cost', 'order_total', 'line_total',
        'base_price', 'market_value', 'asset_value', 'loan_amount', 'credit_limit'
    ],
    'id': [
        'id', 'user_id', 'customer_id', 'order_id', 'employee_id',
        'product_id', 'record_id', 'transaction_id', 'session_id', 'account_id',
        'invoice_id', 'ticket_id', 'item_id', 'batch_id', 'reference_id',
        'uuid', 'guid', 'primary_key', 'row_id', 'seq_id',
        'member_id', 'vendor_id', 'partner_id', 'case_id', 'project_id',
        'category_id', 'department_id', 'branch_id', 'location_id', 'device_id',
        'request_id', 'job_id', 'task_id', 'issue_id', 'event_id',
        'student_id', 'patient_id', 'policy_id', 'claim_id', 'contract_id'
    ],
    'address': [
        'address', 'street', 'city', 'state', 'country',
        'zip_code', 'postal_code', 'street_address', 'home_address', 'office_address',
        'mailing_address', 'billing_address', 'shipping_address', 'location', 'region',
        'province', 'district', 'town', 'neighborhood', 'suburb',
        'zip', 'postcode', 'street_name', 'house_number', 'building',
        'floor', 'apartment', 'suite', 'unit', 'po_box',
        'full_address', 'address_line1', 'address_line2', 'delivery_address', 'resident_address',
        'current_address', 'permanent_address', 'registered_address', 'business_address', 'work_address'
    ],
    'percentage': [
        'percentage', 'percent', 'rate', 'ratio', 'pct',
        'tax_rate', 'discount_rate', 'interest_rate', 'growth_rate', 'success_rate',
        'completion_pct', 'pass_rate', 'fail_rate', 'churn_rate', 'conversion_rate',
        'retention_rate', 'click_rate', 'open_rate', 'bounce_rate', 'engagement_rate',
        'market_share', 'occupancy_rate', 'utilization_rate', 'efficiency_rate', 'accuracy_pct',
        'discount_pct', 'profit_margin', 'gross_margin', 'net_margin', 'tax_pct',
        'ownership_pct', 'allocation_pct', 'savings_rate', 'expense_ratio', 'loss_rate',
        'hit_rate', 'response_rate', 'approval_rate', 'fill_rate', 'coverage_rate'
    ],
    'numeric': [
        'age', 'quantity', 'count', 'score', 'rank',
        'weight', 'height', 'temperature', 'duration', 'distance',
        'volume', 'size', 'length', 'width', 'depth',
        'num_orders', 'total_items', 'page_views', 'clicks', 'impressions',
        'rating', 'stars', 'points', 'level', 'priority',
        'num_employees', 'seat_count', 'room_count', 'floor_count', 'version',
        'sequence_number', 'line_number', 'page_number', 'step_number', 'attempt_count',
        'retry_count', 'error_count', 'warning_count', 'download_count', 'login_count'
    ],
    'text': [
        'notes', 'description', 'comments', 'remarks', 'summary',
        'message', 'content', 'body', 'details', 'info',
        'bio', 'about', 'feedback', 'review', 'testimonial',
        'narrative', 'observation', 'log', 'memo', 'annotation',
        'instructions', 'guidelines', 'terms', 'conditions', 'policy',
        'reason', 'explanation', 'justification', 'diagnosis', 'prescription',
        'question', 'answer', 'response', 'subject', 'title',
        'label', 'tag', 'category_name', 'status_message', 'error_message'
    ]
}

records = []
for label, cols in COLUMN_EXAMPLES.items():
    for col in cols:
        records.append({'column_name': col, 'label': label, 'label_id': LABEL_MAP[label]})

df_data = pd.DataFrame(records).sample(frac=1, random_state=SEED).reset_index(drop=True)
df_data.to_csv('../data/raw/column_labels_dataset.csv', index=False)

print(f'Total examples: {len(df_data)}')
print(f'Per class:\n{df_data["label"].value_counts().to_string()}')

## 2. Train/Test Split

In [ ]:
texts = df_data['column_name'].tolist()
labels = df_data['label_id'].tolist()

X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.20, random_state=SEED, stratify=labels
)

print(f'Train: {len(X_train)} | Test: {len(X_test)}')

## 3. Baseline — Zero-Shot Classification (No Training)

In [ ]:
print('Loading zero-shot classifier (typeform/distilbert-base-uncased-mnli)...')
zero_shot = pipeline(
    'zero-shot-classification',
    model='typeform/distilbert-base-uncased-mnli',
    device=0 if DEVICE == 'cuda' else -1
)

CLASS_NAMES = list(LABEL_MAP.keys())  # ['email', 'phone', ...]

y_pred_baseline = []
for col_name in X_test:
    result = zero_shot(col_name, candidate_labels=CLASS_NAMES)
    # Pick highest-scoring label
    best = result['labels'][0]
    y_pred_baseline.append(LABEL_MAP[best])

baseline_acc = accuracy_score(y_test, y_pred_baseline)
baseline_f1  = f1_score(y_test, y_pred_baseline, average='macro')
print(f'\nBaseline Zero-Shot Accuracy : {baseline_acc:.4f}')
print(f'Baseline Zero-Shot Macro F1 : {baseline_f1:.4f}')
print('\nPer-class report:')
print(classification_report(y_test, y_pred_baseline, target_names=CLASS_NAMES))

## 4. Fine-Tune DistilBERT

In [ ]:
MODEL_NAME = 'distilbert-base-uncased'
tokenizer  = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

class ColNameDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=32):
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=max_len)
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

# Create a small validation set from train set (10%)
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.10, random_state=SEED)

train_ds = ColNameDataset(X_tr,  y_tr,  tokenizer)
val_ds   = ColNameDataset(X_val, y_val, tokenizer)
test_ds  = ColNameDataset(X_test, y_test, tokenizer)

print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1_macro': f1_score(labels, preds, average='macro')
    }

ft_model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=10,
    id2label=ID2LABEL,
    label2id=LABEL_MAP
)

training_args = TrainingArguments(
    output_dir='../models/distilbert-col-labeling',
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    logging_dir='../models/logs',
    logging_steps=10,
    seed=SEED,
    report_to='none'
)

trainer = Trainer(
    model=ft_model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print('Starting fine-tuning...')
train_result = trainer.train()
print('Fine-tuning complete.')
trainer.save_model('../models/distilbert-col-labeling/best')

## 5. Evaluate Fine-Tuned Model

In [ ]:
ft_preds_raw = trainer.predict(test_ds)
y_pred_ft = np.argmax(ft_preds_raw.predictions, axis=-1)

ft_acc = accuracy_score(y_test, y_pred_ft)
ft_f1  = f1_score(y_test, y_pred_ft, average='macro')

print(f'Fine-Tuned Accuracy : {ft_acc:.4f}')
print(f'Fine-Tuned Macro F1 : {ft_f1:.4f}')
print('\nPer-class report:')
print(classification_report(y_test, y_pred_ft, target_names=CLASS_NAMES))

## 6. Before vs After Summary Table

In [ ]:
from sklearn.metrics import classification_report
import json

baseline_report = classification_report(y_test, y_pred_baseline, target_names=CLASS_NAMES, output_dict=True)
ft_report       = classification_report(y_test, y_pred_ft,       target_names=CLASS_NAMES, output_dict=True)

comparison = {
    'model':    ['Zero-Shot DistilBERT (Baseline)', 'Fine-Tuned DistilBERT'],
    'accuracy': [round(baseline_acc * 100, 1),    round(ft_acc * 100, 1)],
    'macro_f1': [round(baseline_f1 * 100, 1),     round(ft_f1 * 100, 1)],
}

for cls in CLASS_NAMES:
    comparison[f'f1_{cls}'] = [
        round(baseline_report[cls]['f1-score'] * 100, 1),
        round(ft_report[cls]['f1-score'] * 100, 1)
    ]

df_compare = pd.DataFrame(comparison)
print(df_compare.to_string(index=False))

results = {
    'baseline_accuracy': round(baseline_acc * 100, 2),
    'baseline_macro_f1': round(baseline_f1 * 100, 2),
    'finetuned_accuracy': round(ft_acc * 100, 2),
    'finetuned_macro_f1': round(ft_f1 * 100, 2),
    'improvement_accuracy': round((ft_acc - baseline_acc) * 100, 2),
    'improvement_f1': round((ft_f1 - baseline_f1) * 100, 2),
    'per_class': {
        cls: {
            'baseline_f1':  round(baseline_report[cls]['f1-score'] * 100, 1),
            'finetuned_f1': round(ft_report[cls]['f1-score'] * 100, 1)
        } for cls in CLASS_NAMES
    }
}
with open('../evaluation/distilbert_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print('\nResults saved to ../evaluation/distilbert_results.json')

## 7. Plots

In [ ]:
# ── Confusion Matrices Side-by-Side ──────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, y_pred, title in zip(
    axes,
    [y_pred_baseline, y_pred_ft],
    ['Zero-Shot Baseline', 'Fine-Tuned DistilBERT']
):
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES)
    disp.plot(ax=ax, colorbar=False, cmap='Blues', xticks_rotation=45)
    ax.set_title(title, fontsize=13, fontweight='bold')

plt.suptitle('Confusion Matrix — Before vs After Fine-Tuning', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../evaluation/distilbert_confusion_matrices.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: distilbert_confusion_matrices.png')

In [ ]:
# ── Per-Class F1 Bar Chart ────────────────────────────────────
x = np.arange(len(CLASS_NAMES))
w = 0.35

baseline_f1s = [baseline_report[c]['f1-score'] * 100 for c in CLASS_NAMES]
ft_f1s       = [ft_report[c]['f1-score'] * 100       for c in CLASS_NAMES]

fig, ax = plt.subplots(figsize=(12, 5))
bars1 = ax.bar(x - w/2, baseline_f1s, w, label='Zero-Shot Baseline', color='#f87171', alpha=0.85)
bars2 = ax.bar(x + w/2, ft_f1s,       w, label='Fine-Tuned',         color='#4ade80', alpha=0.85)

ax.set_xlabel('Semantic Label Class')
ax.set_ylabel('F1 Score (%)')
ax.set_title('Per-Class F1 Score — Zero-Shot vs Fine-Tuned DistilBERT', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES, rotation=30, ha='right')
ax.set_ylim(0, 110)
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.4)

for bar in bars1:
    ax.annotate(f'{bar.get_height():.0f}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=7)
for bar in bars2:
    ax.annotate(f'{bar.get_height():.0f}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=7)

plt.tight_layout()
plt.savefig('../evaluation/distilbert_per_class_f1.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: distilbert_per_class_f1.png')

In [ ]:
# ── Training Loss Curve ───────────────────────────────────────
log_history = trainer.state.log_history
train_logs  = [x for x in log_history if 'loss' in x and 'eval_loss' not in x]
eval_logs   = [x for x in log_history if 'eval_loss' in x]

if train_logs and eval_logs:
    t_steps  = [x['step']      for x in train_logs]
    t_loss   = [x['loss']      for x in train_logs]
    e_epochs = [x['epoch']     for x in eval_logs]
    e_loss   = [x['eval_loss'] for x in eval_logs]
    e_acc    = [x.get('eval_accuracy', 0) * 100 for x in eval_logs]
    e_f1     = [x.get('eval_f1_macro', 0) * 100 for x in eval_logs]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].plot(t_steps, t_loss, color='#6366f1', linewidth=1.5)
    axes[0].set_title('Training Loss'); axes[0].set_xlabel('Step'); axes[0].set_ylabel('Loss')
    axes[0].grid(linestyle='--', alpha=0.4)

    axes[1].plot(e_epochs, e_acc, marker='o', color='#f59e0b', linewidth=2)
    axes[1].set_title('Validation Accuracy (%)'); axes[1].set_xlabel('Epoch')
    axes[1].set_ylim(0, 100); axes[1].grid(linestyle='--', alpha=0.4)

    axes[2].plot(e_epochs, e_f1, marker='s', color='#10b981', linewidth=2)
    axes[2].set_title('Validation Macro F1 (%)'); axes[2].set_xlabel('Epoch')
    axes[2].set_ylim(0, 100); axes[2].grid(linestyle='--', alpha=0.4)

    plt.suptitle('DistilBERT Fine-Tuning Progress', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../evaluation/distilbert_training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: distilbert_training_curves.png')
else:
    print('No log history found — rerun with logging_steps=1 if needed.')

## 8. Final Summary

In [ ]:
print('=' * 55)
print('          DISTILBERT RESULTS SUMMARY')
print('=' * 55)
print(f'  Zero-Shot Baseline  →  Accuracy: {results["baseline_accuracy"]:5.1f}%   F1: {results["baseline_macro_f1"]:5.1f}%')
print(f'  Fine-Tuned Model    →  Accuracy: {results["finetuned_accuracy"]:5.1f}%   F1: {results["finetuned_macro_f1"]:5.1f}%')
print(f'  Improvement         →  Accuracy: +{results["improvement_accuracy"]:4.1f}pp   F1: +{results["improvement_f1"]:4.1f}pp')
print('=' * 55)
print(f'  Model saved to: ../models/distilbert-col-labeling/best')
print(f'  Plots saved to: ../evaluation/')
print('=' * 55)